In [1]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
from pathlib import Path
import logging
from io import BytesIO
from apicall_input import main_api_call 

In [14]:
def create_emptydf(start_date,end_date):
    """
    Creates empty DataFrame with date range
    Args:
        start_date (str): Start date in 'yyyy-mm-dd' format
        end_date (str): End date in 'yyyy-mm-dd' format
        
    Returns:
        empty (df): Eempty df ready for population
    """
    start_date = str(start_date)
    end_date = str(end_date)
    # Convert start_date to a consistent format
    start_date = start_date.split(' ')[0]  # Remove time if present
    start = datetime.strptime(start_date, '%Y-%m-%d')

    end_date = end_date.split(' ')[0]  # Remove time if present
    end = datetime.strptime(end_date, '%Y-%m-%d')
    
    date_range = pd.date_range(start, end)

    df = pd.DataFrame({'Date': date_range})
    
    df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
    df['nr. sessions'] = 0
    df['total km'] = 0.00
    df['km Z3-4'] = 0.00
    df['km Z5-T1-T2'] = 0.00
    df['hours alternative'] = 0.00
    return df

In [15]:
def convert_to_day_approach(df):
    """
    Converts the DataFrame to a day approach format.
    
    Args:
        df (DataFrame): The DataFrame to convert.
        
    Returns:
        DataFrame: The converted DataFrame into a format with 7 lagging days 
        before each date in the format 

    """
    feature_cols = ['nr. sessions', 'total km', 'km Z3-4', 'km Z5-T1-T2', 'hours alternative']
    df_converted = pd.DataFrame()
    for i in range(0,7):
        for col in feature_cols:
            df_converted[f'{col}.{i}'] = df[col].shift(i)  
    df_converted['Date'] = df['Date']
    # drop rows with NaN values using dropna() with index as the row
    df_converted = df_converted.dropna()

    # replace the name of the column with the name of the column without the last 2 characters
    df_converted = df_converted.rename(columns={col: col[:-2] for col in df_converted.columns if col.endswith('.0')})


    # return df_lagged
    return df_converted 

In [16]:
def readfiles(df_dict):
    running_activities = []
    other_activities = []

    # Regex pattern for filenames containing 'running_'
    # re.compile pre-compiles the regex for efficiency if used in a loop
    running_pattern = re.compile(r".*running.*\.csv$", re.IGNORECASE)

    # Iterate through each dictionary in the list
    for data_dict in df_dict:
        filename = data_dict["filename"]
    
    # Check if the filename matches the regex pattern
        if running_pattern.match(filename):
            running_activities.append(data_dict)
        else:
            other_activities.append(data_dict)
   

    return running_activities,other_activities

In [17]:
def populateone_memory(df_prepop, file_df, filedate, Z3_min, Z5_min):
    """
    Populates the empty DataFrame with the data from the file
    Args:
        df_prepop (df): DataFrame to be populated
        file_df (df): DataFrame containing activity data
        filedate (str): Date of the activity
        Z3_min (int): Minimum heart rate for Z3 zone
        Z5_min (int): Minimum heart rate for Z5 zone
    Returns:
        df_postpop (df): Populated DataFrame
    """
    file_df['Distance'] = pd.to_numeric(file_df['Distance'], errors='coerce')

    df_prepop.loc[df_prepop['Date'] == filedate, 'total km'] += file_df['Distance'].iloc[-1]

    for idx, row in file_df.iloc[:-1].iterrows():
        hr = row['Avg HR']
        distance = row['Distance']
        if Z3_min <= hr < Z5_min:
            df_prepop.loc[df_prepop['Date'] == filedate, 'km Z3-4'] += distance
        elif hr >= Z5_min:
            df_prepop.loc[df_prepop['Date'] == filedate, 'km Z5-T1-T2'] += distance

    return df_prepop

In [18]:
def populatebydate_memory(emptydf, run_activities, other_activities, Z3_min, Z5_min):
    for i in emptydf['Date']:
        for activity in run_activities:
            filedate = datetime.strptime(activity['filename'].split('|')[1], '%d-%m-%Y').strftime('%Y-%m-%d')
            if filedate == i:
                emptydf.loc[emptydf['Date'] == filedate, 'nr. sessions'] += 1
                populateone_memory(emptydf, activity['df'], filedate, Z3_min, Z5_min)

        for activity in other_activities:
            filedate = datetime.strptime(activity['filename'].split('|')[1], '%d-%m-%Y').strftime('%Y-%m-%d')
            if filedate == i:
                temp_df = activity['df']
                time_str = temp_df['Time'].iloc[-1]
                time_obj = datetime.strptime(time_str, '%H:%M:%S.%f').time()
                time_delta = timedelta(hours=time_obj.hour, minutes=time_obj.minute, seconds=time_obj.second, microseconds=time_obj.microsecond)

                hours_alternative = round(time_delta.total_seconds() / 3600, 2)
                emptydf.loc[emptydf['Date'] == filedate, 'hours alternative'] = hours_alternative

    return emptydf

In [ ]:
def initial_transform(start_date, end_date, df_memory, Z3_min = 135, Z5_min = 173):   
    """
    Main function to extract and transform data.
    """
    try:
        Z3_min = int(Z3_min)
        Z5_min = int(Z5_min)
    except ValueError:
        print("Please enter valid numbers for heart rate zone thresholds.")

   
    empty = create_emptydf(start_date, end_date)
    r,o = readfiles(df_memory)
    
    df_full = populatebydate_memory(empty, r, o, Z3_min, Z5_min)
    
    # Convert to day approach format
    dfday_user = convert_to_day_approach(df_full)
    
    return dfday_user

In [ ]:
def refactor(df):
    
    return df

In [21]:
def main_extract_transform_memory(start_date, end_date, df_memory, Z3_min = 135, Z5_min = 173):
    df = initial_transform(start_date, end_date, df_memory, Z3_min, Z5_min)
    refactored_df = refactor(df)


    print(refactored_df)
    return refactored_df


# Testing

In [10]:
start_date, end_date, df_memory = main_api_call()


Garmin Connect API - Activity Downloader
Login successful!
Activity data for 'indoor_cardio|12-08-2025|20034672816.csv' loaded into DataFrame.
Activity data for 'walking|10-08-2025|20012275661.csv' loaded into DataFrame.
Activity data for 'running|10-08-2025|20010223403.csv' loaded into DataFrame.
Activity data for 'treadmill_running|08-08-2025|19990936196.csv' loaded into DataFrame.
Activity data for 'indoor_cardio|07-08-2025|19983834305.csv' loaded into DataFrame.
Activity data for 'running|06-08-2025|19966811412.csv' loaded into DataFrame.
Activity data for 'indoor_cardio|05-08-2025|19957680422.csv' loaded into DataFrame.
Activity data for 'indoor_cardio|04-08-2025|19948544911.csv' loaded into DataFrame.
Activity data for 'running|03-08-2025|19933730154.csv' loaded into DataFrame.
Activity data for 'running|01-08-2025|19918398512.csv' loaded into DataFrame.
Activity data for 'indoor_cardio|31-07-2025|19909024936.csv' loaded into DataFrame.
Activity data for 'running|30-07-2025|19892

In [22]:
main_extract_transform_memory(start_date, end_date, df_memory)

    nr. sessions  total km  km Z3-4  km Z5-T1-T2  hours alternative  \
6              0      0.00     0.00         0.00               0.00   
7              0      0.00     0.00         0.00               0.00   
8              0      0.00     0.00         0.00               0.00   
9              0      0.00     0.00         0.00               0.00   
10             0      0.00     0.00         0.00               0.55   
11             0      0.00     0.00         0.00               0.00   
12             0      0.00     0.00         0.00               0.56   
13             0      0.00     0.00         0.00               0.00   
14             1      3.00     3.00         0.00               0.00   
15             0      0.00     0.00         0.00               0.81   
16             1      5.09     2.09         0.00               0.00   
17             0      0.00     0.00         0.00               0.00   
18             1      3.23     1.23         2.00               0.32   
19    

,nr. sessions,total km,km Z3-4,km Z5-T1-T2,hours alternative,nr. sessions.1,total km.1,km Z3-4.1,km Z5-T1-T2.1,hours alternative.1,...,total km.5,km Z3-4.5,km Z5-T1-T2.5,hours alternative.5,nr. sessions.6,total km.6,km Z3-4.6,km Z5-T1-T2.6,hours alternative.6,Date
6,0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.67,...,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,2025-06-19
7,0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,...,1.50,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,2025-06-20
8,0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,1.0,1.50,0.00,0.00,0.00,2025-06-21
9,0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,2025-06-22
10,0,0.00,0.00,0.00,0.55,0.0,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.67,0.0,0.00,0.00,0.00,0.00,2025-06-23
11,0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.55,...,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.67,2025-06-24
12,0,0.00,0.00,0.00,0.56,0.0,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,2025-06-25
13,0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.56,...,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,2025-06-26
14,1,3.00,3.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,2025-06-27
15,0,0.00,0.00,0.00,0.81,1.0,3.00,3.00,0.00,0.00,...,0.00,0.00,0.00,0.55,0.0,0.00,0.00,0.00,0.00,2025-06-28


In [ ]:

def create_combodf(df):
    """
    Create a DataFrame with day-based aggregations.
    """
    combodf = pd.DataFrame()
    combodf['Date'] = df['Date']
    combodf['Day1 total km'] = df['total km']
    combodf['Day1 km z3+'] = df['km Z3-4'] + df['km Z5-T1-T2']
    combodf['Day1 km z5'] = df['km Z5-T1-T2']
    combodf['Day2-3 nr.sessions'] = df['nr. sessions.1'] + df['nr. sessions.2']
    combodf['Day2-3 total km'] = df['total km.1'] + df['total km.2']
    combodf['Day2-3 km z3+'] = (
        df['km Z3-4.1'] + df['km Z3-4.2'] + df['km Z5-T1-T2.1'] + df['km Z5-T1-T2.2']
    )
    combodf['Day2-3 km z5'] = df['km Z5-T1-T2.1'] + df['km Z5-T1-T2.2']
    combodf['Day4-7 nr.sessions'] = (
        df['nr. sessions.3'] + df['nr. sessions.4'] + df['nr. sessions.5'] + df['nr. sessions.6']
    )
    combodf['Day4-7 total km'] = (
        df['total km.3'] + df['total km.4'] + df['total km.5'] + df['total km.6']
    )
    combodf['Day4-7 km z3+'] = (
        df['km Z3-4.3'] + df['km Z3-4.4'] + df['km Z3-4.5'] + df['km Z3-4.6'] +
        df['km Z5-T1-T2.3'] + df['km Z5-T1-T2.4'] + df['km Z5-T1-T2.5'] + df['km Z5-T1-T2.6']
    )
    return combodf

def create_weekly_df(df):
    """
    Create a DataFrame with week-based aggregations.
    """
    weekly_df = pd.DataFrame()
    weekly_df['Date'] = df['Date']
    weekly_df['Week1 max km one day'] = (
        df[['total km', 'total km.1', 'total km.2', 'total km.3', 'total km.4', 'total km.5', 'total km.6']]
        .shift(7).max(axis=1)
    )
    weekly_df['Week1 total km z3+'] = (
        df[['km Z3-4', 'km Z3-4.1', 'km Z3-4.2', 'km Z3-4.3', 'km Z3-4.4', 'km Z3-4.5', 'km Z3-4.6']]
        .shift(7).sum(axis=1) +
        df[['km Z5-T1-T2', 'km Z5-T1-T2.1', 'km Z5-T1-T2.2', 'km Z5-T1-T2.3', 'km Z5-T1-T2.4', 'km Z5-T1-T2.5', 'km Z5-T1-T2.6']]
        .shift(7).sum(axis=1)
    )
    weekly_df['Week1 max km Z3+ one day'] = (
        df[['km Z3-4', 'km Z3-4.1', 'km Z3-4.2', 'km Z3-4.3', 'km Z3-4.4', 'km Z3-4.5', 'km Z3-4.6']]
        .shift(7).max(axis=1) +
        df[['km Z5-T1-T2', 'km Z5-T1-T2.1', 'km Z5-T1-T2.2', 'km Z5-T1-T2.3', 'km Z5-T1-T2.4', 'km Z5-T1-T2.5', 'km Z5-T1-T2.6']]
        .shift(7).max(axis=1)
    )
    # Repeat for Week2
    weekly_df['Week2 max km one day'] = (
        df[['total km', 'total km.1', 'total km.2', 'total km.3', 'total km.4', 'total km.5', 'total km.6']]
        .shift(14).max(axis=1)
    )
    weekly_df['Week2 total km z3+'] = (
        df[['km Z3-4', 'km Z3-4.1', 'km Z3-4.2', 'km Z3-4.3', 'km Z3-4.4', 'km Z3-4.5', 'km Z3-4.6']]
        .shift(14).sum(axis=1) +
        df[['km Z5-T1-T2', 'km Z5-T1-T2.1', 'km Z5-T1-T2.2', 'km Z5-T1-T2.3', 'km Z5-T1-T2.4', 'km Z5-T1-T2.5', 'km Z5-T1-T2.6']]
        .shift(14).sum(axis=1)
    )
    weekly_df['Week2 max km Z3+ one day'] = (
        df[['km Z3-4', 'km Z3-4.1', 'km Z3-4.2', 'km Z3-4.3', 'km Z3-4.4', 'km Z3-4.5', 'km Z3-4.6']]
        .shift(14).max(axis=1) +
        df[['km Z5-T1-T2', 'km Z5-T1-T2.1', 'km Z5-T1-T2.2', 'km Z5-T1-T2.3', 'km Z5-T1-T2.4', 'km Z5-T1-T2.5', 'km Z5-T1-T2.6']]
        .shift(14).max(axis=1)
    )
    weekly_df.dropna(inplace=True)
    return weekly_df

def calculate_ratios_and_acwr(newdf, weekly_df, combodf):
    """
    Calculate ratios and ACWR.
    """
    # Week 0 (current week)
    Week0_total_km = newdf[['total km', 'total km.1', 'total km.2', 'total km.3', 'total km.4', 'total km.5', 'total km.6']].sum(axis=1)
    Week0_hours_alternative = newdf[['hours alternative', 'hours alternative.1', 'hours alternative.2', 'hours alternative.3', 'hours alternative.4', 'hours alternative.5', 'hours alternative.6']].sum(axis=1)

    # Week 1 and Week 2
    Week1_total_km = weekly_df['Week1 total km z3+']
    Week2_total_km = weekly_df['Week2 total km z3+']

    # Ratios
    day5totkm = newdf[['total km', 'total km.1', 'total km.2', 'total km.3', 'total km.4']].sum(axis=1)
    week3totkm = Week0_total_km + Week1_total_km + Week2_total_km
    combodf['5day/3W tot km ratio'] = np.where(week3totkm != 0, (day5totkm / week3totkm).round(3), 0)

    # ACWR
    combodf['ACWR'] = np.where(
        week3totkm != 0,
        ((day5totkm * 4) / week3totkm).round(3),
        0
    )
    return combodf

def main_transformations(df):
    """
    Main function to perform all transformations.
    """
    combodf = create_combodf(df)
    weekly_df = create_weekly_df(df)
    combodf = pd.merge(combodf, weekly_df, on='Date', how='inner')
    newdf = df[df['Date'].isin(weekly_df['Date'])]
    combodf = calculate_ratios_and_acwr(newdf, weekly_df, combodf)
    return combodf